<a href="https://colab.research.google.com/github/NhanChanDeo/VQA_Final_Thesis/blob/main/DATASETS%20SCHEMA%20NORMALIZATION/Step9_Normalize_CrossVQA_Parquet_lossless_branch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 9: Streaming Sharded Normalization & Lossless Parquet Deployment (CrossVQA)

This notebook normalizes **TextVQA** (`gamusa/textvqa-v0.5.1`) and **DocVQA** (`gamusa/docvqa-task1-spdocvqa`) into the **Canonical CrossVQA Schema**, retaining original image formats and deploying to the **`lossless` branch** on Hugging Face.

### Key Features & Architecture:
1. **On-the-Fly Streaming Sharding (Zero OOM / Constant < 250MB RAM):**
   - Eliminates monolithic files that trigger `TooBigContentError` (HTTP 501 / Scan size limit exceeded) on Hugging Face Dataset Viewer.
   - Streams data directly to disk using `pyarrow.parquet.ParquetWriter` with `write_page_index=True` for instant DuckDB random seeking.
   - **TextVQA Shards:** 2,500 samples/shard (~350 MB each, 14 train shards).
   - **DocVQA Shards:** 800 samples/shard (~800MB–1.1 GB each, 50 train shards) — perfectly sized for DuckDB and Hugging Face streaming.
2. **Lossless Format & EXIF Orientation Preservation:**
   - **DocVQA:** Preserves 100% original uncompressed **PNG** bytes from scanned documents (zero lossy compression artifacts).
   - **TextVQA:** Preserves native **JPEG** bytes bit-for-bit. If EXIF orientation tag is present and != 1, applies `ImageOps.exif_transpose` and saves at Q100 (4:4:4 subsampling) so visual coordinates strictly align with Rosetta OCR annotations; all upright images preserve raw bytes without re-compression generational loss.
3. **Automatic Hub & Local Cleanup:**
   - **Hub Atomic Pruning:** Detects and prunes obsolete shards inherited from `main` (e.g. DocVQA's 25 `*-of-00027.parquet` JPEG shards) in a single atomic commit via `CommitOperationDelete`, preventing file collisions and duplicate rows in Dataset Viewer.
   - **Disk Quota Protection:** Automatically removes the 8.6 GB raw tar archive after extraction to keep Colab disk headroom safe throughout.
4. **Hub Deployment:** Uses multi-threaded `api.upload_folder` directly to the **`lossless` branch** (`gamusa/textvqa-canonical` and `gamusa/docvqa-canonical`).

In [2]:
# 1. Environment Setup & HF Authentication
!pip install -q --upgrade pip
!pip install -q --upgrade huggingface_hub datasets pyarrow pandas tqdm Pillow

import os
import gc
import io
import math
import json
import shutil
import zipfile
import tarfile
import subprocess
from collections import Counter
from pathlib import Path
from typing import List, Dict, Any, Optional
from PIL import Image, ImageOps
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from huggingface_hub import HfApi, login, hf_hub_download, CommitOperationDelete

# Dual-environment Token Auth (Colab userdata + local env fallback)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("Please configure HF_TOKEN in Colab Secrets or environment variables!")

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
print(f"✓ Authenticated as: {username}")

# Target branch for lossless datasets
TARGET_BRANCH = "lossless"
print(f"✓ Target Hub Branch: {TARGET_BRANCH}")

# Workspace paths
WORK_DIR = Path("./crossvqa_lossless_workspace")
RAW_DIR = WORK_DIR / "raw"
SHARD_DIR = WORK_DIR / "shards"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.5 MB/s eta 0:00:00
✓ Authenticated as: gamusa
✓ Target Hub Branch: lossless


In [3]:
# 2. Canonical PyArrow Schema & Streaming Sharding Engine

# Strict canonical PyArrow schema for uniform CrossVQA shards
CANONICAL_PYARROW_SCHEMA = pa.schema([
    ("sample_id", pa.string()),
    ("domain_type", pa.string()),
    ("split", pa.string()),
    ("question_id", pa.string()),
    ("question", pa.string()),
    ("question_tokens", pa.list_(pa.string())),
    ("image_id", pa.string()),
    ("image", pa.struct([("bytes", pa.binary()), ("path", pa.string())])),
    ("image_width", pa.int64()),
    ("image_height", pa.int64()),
    ("answers", pa.list_(pa.string())),
    ("canonical_answer", pa.string()),
    ("ocr_tokens", pa.list_(pa.string())),
    ("ocr_boxes", pa.list_(pa.list_(pa.int64()))),
    ("metadata", pa.string()),
])

def extract_canonical_answer(answers: Optional[List[str]]) -> Optional[str]:
    if not answers:
        return None
    cleaned = [ans.strip() for ans in answers if ans and ans.strip()]
    if not cleaned:
        return None
    return Counter(cleaned).most_common(1)[0][0]

def normalize_rosetta_box(box_dict: Dict[str, float]) -> List[int]:
    tl_x = box_dict.get("top_left_x", 0.0)
    tl_y = box_dict.get("top_left_y", 0.0)
    w = box_dict.get("width", 0.0)
    h = box_dict.get("height", 0.0)
    xmin = int(round(max(0.0, min(1.0, tl_x)) * 1000))
    ymin = int(round(max(0.0, min(1.0, tl_y)) * 1000))
    xmax = int(round(max(0.0, min(1.0, tl_x + w)) * 1000))
    ymax = int(round(max(0.0, min(1.0, tl_y + h)) * 1000))
    return [ymin, xmin, ymax, xmax]

def normalize_ms_ocr_polygon(polygon: List[float], img_w: int, img_h: int) -> List[int]:
    if not polygon or len(polygon) < 8 or img_w <= 0 or img_h <= 0:
        return [0, 0, 0, 0]
    xs = polygon[0::2]
    ys = polygon[1::2]
    xmin = int(round(max(0.0, min(1.0, min(xs) / img_w)) * 1000))
    xmax = int(round(max(0.0, min(1.0, max(xs) / img_w)) * 1000))
    ymin = int(round(max(0.0, min(1.0, min(ys) / img_h)) * 1000))
    ymax = int(round(max(0.0, min(1.0, max(ys) / img_h)) * 1000))
    return [ymin, xmin, ymax, xmax]

class StreamingParquetSharder:
    """
    Streams rows directly to Parquet shards with constant RAM footprint (< 250MB).
    Enables page index for DuckDB and Hugging Face Dataset Viewer compatibility.
    """
    def __init__(
        self,
        output_dir: Path,
        split_name: str,
        total_rows: int,
        shard_size: int,
        schema: pa.Schema = CANONICAL_PYARROW_SCHEMA,
        compression: str = 'zstd'
    ):
        self.output_dir = output_dir
        self.split_name = split_name
        self.total_rows = total_rows
        self.shard_size = shard_size
        self.schema = schema
        self.compression = compression
        self.total_shards = max(1, math.ceil(total_rows / shard_size))
        self.current_shard_idx = 0
        self.rows_in_current_shard = 0
        self.writer = None
        self.created_shards = []
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self._open_new_shard()

    def _open_new_shard(self):
        if self.writer is not None:
            self.writer.close()
            p = self.created_shards[-1]
            print(f"  ✓ Shard {self.current_shard_idx}/{self.total_shards}: {p.name} ({p.stat().st_size / (1024**2):.1f} MB)")

        shard_name = f"{self.split_name}-{self.current_shard_idx:05d}-of-{self.total_shards:05d}.parquet"
        shard_path = self.output_dir / shard_name
        self.writer = pq.ParquetWriter(
            shard_path,
            self.schema,
            compression=self.compression,
            write_page_index=True  # Enables DuckDB fast page-level random access on HF Viewer
        )
        self.created_shards.append(shard_path)
        self.rows_in_current_shard = 0
        self.current_shard_idx += 1

    def write_rows(self, rows: List[Dict[str, Any]]):
        if not rows:
            return
        offset = 0
        while offset < len(rows):
            remaining = self.shard_size - self.rows_in_current_shard
            chunk = rows[offset : offset + remaining]
            table = pa.Table.from_pylist(chunk, schema=self.schema)
            self.writer.write_table(table)
            self.rows_in_current_shard += len(chunk)
            offset += len(chunk)

            if self.rows_in_current_shard >= self.shard_size and self.current_shard_idx < self.total_shards:
                self._open_new_shard()

    def close(self) -> List[Path]:
        if self.writer is not None:
            self.writer.close()
            p = self.created_shards[-1]
            print(f"  ✓ Shard {self.current_shard_idx}/{self.total_shards}: {p.name} ({p.stat().st_size / (1024**2):.1f} MB)")
            self.writer = None
        return self.created_shards


In [4]:
# 3. Download TextVQA Assets from Hub
TEXTVQA_REPO = f"{username}/textvqa-v0.5.1"
print(f"Downloading TextVQA assets from {TEXTVQA_REPO}...")

textvqa_files = [
    "annotations/TextVQA_0.5.1_train.json",
    "annotations/TextVQA_0.5.1_val.json",
    "annotations/TextVQA_0.5.1_test.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_train.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_val.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_test.json",
    "images/train_val_images.zip",
    "images/test_images.zip",
]

textvqa_paths = {}
for rfile in textvqa_files:
    textvqa_paths[rfile] = hf_hub_download(
        repo_id=TEXTVQA_REPO,
        repo_type="dataset",
        filename=rfile,
        local_dir=str(RAW_DIR / "textvqa")
    )
print("✓ TextVQA assets downloaded successfully.")


annotations/TextVQA_0.5.1_train.json: reconstructing file:   0%|          |  0.00B / 21.6MB            

annotations/TextVQA_0.5.1_train.json: downloading bytes:           |  0.00B            

TextVQA_0.5.1_val.json:   0%|          | 0.00/3.12M [00:00<?, ?B/s]

TextVQA_0.5.1_test.json:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

ocr/TextVQA_Rosetta_OCR_v0.2_train.json: reconstructing file:   0%|          |  0.00B / 65.8MB            

...

images/train_val_images.zip: reconstructing file:   0%|          |  0.00B / 7.07GB            

images/train_val_images.zip: downloading bytes:           |  0.00B            

images/test_images.zip: reconstructing file:   0%|          |  0.00B /  970MB            

images/test_images.zip: downloading bytes:           |  0.00B            

✓ TextVQA assets downloaded successfully.


In [5]:
# 4. Stream & Shard TextVQA (Lossless / Native JPEG with EXIF Awareness)
TEXTVQA_SHARD_DIR = SHARD_DIR / "textvqa"
TEXTVQA_SHARD_DIR.mkdir(parents=True, exist_ok=True)

# 2,500 samples per shard keeps files ~300-380 MB
TEXTVQA_SHARD_SIZE = 2500

def process_textvqa_streaming(split_name, ann_key, ocr_key, zip_key):
    ann_path = textvqa_paths[ann_key]
    ocr_path = textvqa_paths[ocr_key]
    zip_path = textvqa_paths[zip_key]

    with open(ann_path, 'r', encoding='utf-8') as f:
        items = json.load(f)['data']
    with open(ocr_path, 'r', encoding='utf-8') as f:
        ocr_raw = json.load(f)['data']

    ocr_lookup = {}
    for o in ocr_raw:
        tokens = o.get('ocr_tokens', [])
        boxes = [normalize_rosetta_box(b.get('bounding_box', {})) for b in o.get('ocr_info', [])]
        ocr_lookup[o['image_id']] = {'tokens': tokens, 'boxes': boxes}

    zip_ref = zipfile.ZipFile(zip_path, 'r')
    prefix = "train_images/" if split_name != "test" else "test_images/"

    sharder = StreamingParquetSharder(
        output_dir=TEXTVQA_SHARD_DIR,
        split_name=split_name,
        total_rows=len(items),
        shard_size=TEXTVQA_SHARD_SIZE,
        compression="zstd"
    )

    batch_buffer = []
    BUFFER_FLUSH_SIZE = 100

    print(f"\n{'='*20} Sharding TextVQA [{split_name}]: {len(items)} rows {'='*20}")
    for item in tqdm(items, desc=f"TextVQA [{split_name}]"):
        q_id = str(item['question_id'])
        img_id = item['image_id']
        img_name = f"{prefix}{img_id}.jpg"

        raw_bytes = zip_ref.read(img_name)

        # EXIF Awareness
        with Image.open(io.BytesIO(raw_bytes)) as img:
            exif = img.getexif()
            orientation = exif.get(0x0112, 1) if exif else 1

            if orientation not in (1, 0, None):
                transposed = ImageOps.exif_transpose(img)
                orig_w, orig_h = transposed.size
                buf = io.BytesIO()
                transposed.save(buf, format="JPEG", quality=100, subsampling=0)
                final_bytes = buf.getvalue()
            else:
                orig_w, orig_h = img.size
                final_bytes = raw_bytes

        ocr_entry = ocr_lookup.get(img_id, {'tokens': [], 'boxes': []})
        answers = item.get('answers', None)

        batch_buffer.append({
            "sample_id": f"textvqa_{q_id}",
            "domain_type": "scene_text",
            "split": split_name,
            "question_id": q_id,
            "question": item['question'].strip(),
            "question_tokens": item.get('question_tokens', []),
            "image_id": img_id,
            "image": {'bytes': final_bytes, 'path': f"{img_id}.jpg"},
            "image_width": orig_w,
            "image_height": orig_h,
            "answers": answers if answers else [],
            "canonical_answer": extract_canonical_answer(answers),
            "ocr_tokens": ocr_entry['tokens'],
            "ocr_boxes": ocr_entry['boxes'],
            "metadata": json.dumps({
                "image_classes": item.get('image_classes', []),
                "flickr_original_url": item.get('flickr_original_url', ''),
                "exif_transposed": bool(orientation not in (1, 0, None))
            }, ensure_ascii=False)
        })

        if len(batch_buffer) >= BUFFER_FLUSH_SIZE:
            sharder.write_rows(batch_buffer)
            batch_buffer.clear()

    if batch_buffer:
        sharder.write_rows(batch_buffer)
        batch_buffer.clear()

    zip_ref.close()
    shards = sharder.close()
    gc.collect()
    pa.default_memory_pool().release_unused()
    print(f"✓ Completed TextVQA [{split_name}]: {len(shards)} shards created.")

process_textvqa_streaming("train", "annotations/TextVQA_0.5.1_train.json", "ocr/TextVQA_Rosetta_OCR_v0.2_train.json", "images/train_val_images.zip")
process_textvqa_streaming("validation", "annotations/TextVQA_0.5.1_val.json", "ocr/TextVQA_Rosetta_OCR_v0.2_val.json", "images/train_val_images.zip")
process_textvqa_streaming("test", "annotations/TextVQA_0.5.1_test.json", "ocr/TextVQA_Rosetta_OCR_v0.2_test.json", "images/test_images.zip")



==================== Sharding TextVQA [train]: 34602 rows ====================


TextVQA [train]:   0%|          | 0/34602 [00:00<?, ?it/s]

  ✓ Shard 1/14: train-00000-of-00014.parquet (413.5 MB)
  ✓ Shard 2/14: train-00001-of-00014.parquet (403.1 MB)
  ✓ Shard 3/14: train-00002-of-00014.parquet (413.6 MB)
  ✓ Shard 4/14: train-00003-of-00014.parquet (406.8 MB)
  ✓ Shard 5/14: train-00004-of-00014.parquet (402.8 MB)
  ✓ Shard 6/14: train-00005-of-00014.parquet (342.1 MB)
  ✓ Shard 7/14: train-00006-of-00014.parquet (407.6 MB)
  ✓ Shard 8/14: train-00007-of-00014.parquet (453.2 MB)
  ✓ Shard 9/14: train-00008-of-00014.parquet (511.6 MB)
  ✓ Shard 10/14: train-00009-of-00014.parquet (506.9 MB)
  ✓ Shard 11/14: train-00010-of-00014.parquet (413.8 MB)
  ✓ Shard 12/14: train-00011-of-00014.parquet (451.0 MB)
  ✓ Shard 13/14: train-00012-of-00014.parquet (341.1 MB)
  ✓ Shard 14/14: train-00013-of-00014.parquet (453.7 MB)
✓ Completed TextVQA [train]: 14 shards created.

==================== Sharding TextVQA [validation]: 5000 rows ====================


TextVQA [validation]:   0%|          | 0/5000 [00:00<?, ?it/s]

  ✓ Shard 1/2: validation-00000-of-00002.parquet (450.3 MB)
  ✓ Shard 2/2: validation-00001-of-00002.parquet (427.6 MB)
✓ Completed TextVQA [validation]: 2 shards created.

==================== Sharding TextVQA [test]: 5734 rows ====================


TextVQA [test]:   0%|          | 0/5734 [00:00<?, ?it/s]

  ✓ Shard 1/3: test-00000-of-00003.parquet (406.7 MB)
  ✓ Shard 2/3: test-00001-of-00003.parquet (403.9 MB)
  ✓ Shard 3/3: test-00002-of-00003.parquet (110.2 MB)
✓ Completed TextVQA [test]: 3 shards created.


In [6]:
# 5. Download DocVQA Assets from Hub
DOCVQA_REPO = f"{username}/docvqa-task1-spdocvqa"
print(f"Downloading DocVQA assets from {DOCVQA_REPO}...")

docvqa_files = [
    "annotations/train_v1.0_withQT.json",
    "annotations/val_v1.0_withQT.json",
    "annotations/test_v1.0.json",
    "images/spdocvqa_images.tar.gz",
    "ocr/spdocvqa_ocr.tar.gz"
]

docvqa_paths = {}
for rfile in docvqa_files:
    docvqa_paths[rfile] = hf_hub_download(
        repo_id=DOCVQA_REPO,
        repo_type="dataset",
        filename=rfile,
        local_dir=str(RAW_DIR / "docvqa")
    )
print("✓ DocVQA assets downloaded successfully.")


annotations/train_v1.0_withQT.json: reconstructing file:   0%|          |  0.00B / 11.6MB            

annotations/train_v1.0_withQT.json: downloading bytes:           |  0.00B            

val_v1.0_withQT.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

test_v1.0.json:   0%|          | 0.00/1.15M [00:00<?, ?B/s]

images/spdocvqa_images.tar.gz: reconstructing file:   0%|          |  0.00B / 8.58GB            

images/spdocvqa_images.tar.gz: downloading bytes:           |  0.00B            

ocr/spdocvqa_ocr.tar.gz: reconstructing file:   0%|          |  0.00B / 62.9MB            

ocr/spdocvqa_ocr.tar.gz: downloading bytes:           |  0.00B            

✓ DocVQA assets downloaded successfully.


In [7]:
# 6. Stream & Shard DocVQA (Lossless PNG with EXIF Awareness & Disk Protection)
DOCVQA_SHARD_DIR = SHARD_DIR / "docvqa"
DOCVQA_SHARD_DIR.mkdir(parents=True, exist_ok=True)

# 1. Sequential linear extraction to disk (one-time linear pass)
DOC_IMG_DIR = RAW_DIR / "docvqa_extracted_images"
DOC_OCR_DIR = RAW_DIR / "docvqa_extracted_ocr"
DOC_IMG_DIR.mkdir(parents=True, exist_ok=True)
DOC_OCR_DIR.mkdir(parents=True, exist_ok=True)

if not any(DOC_IMG_DIR.iterdir()):
    print("Extracting images sequentially to disk (one-time linear pass)...")
    subprocess.run([
        "tar", "-xzf", str(docvqa_paths["images/spdocvqa_images.tar.gz"]),
        "-C", str(DOC_IMG_DIR)
    ], check=True)

    # DISK PROTECTION: Reclaim 8.6 GB immediately by deleting archive after extraction
    tar_archive = Path(docvqa_paths["images/spdocvqa_images.tar.gz"])
    if tar_archive.exists():
        tar_archive.unlink()
        print(f"  🗑️ Deleted {tar_archive.name} to reclaim 8.6 GB disk space.")

if not any(DOC_OCR_DIR.iterdir()):
    print("Extracting OCR sequentially to disk (one-time linear pass)...")
    subprocess.run([
        "tar", "-xzf", str(docvqa_paths["ocr/spdocvqa_ocr.tar.gz"]),
        "-C", str(DOC_OCR_DIR)
    ], check=True)

# 800 samples/shard produces ~800MB - 1.1GB per shard (optimal for DuckDB & HF Viewer)
DOCVQA_SHARD_SIZE = 800

def process_docvqa_streaming(split_name, ann_key):
    ann_path = docvqa_paths[ann_key]

    with open(ann_path, 'r', encoding='utf-8') as f:
        items = json.load(f)['data']

    # Cache metadata only (w, h, orientation) in RAM to keep RAM < 150MB throughout run
    image_dim_cache = {}  # img_basename -> (orig_w, orig_h, orientation)
    ocr_cache = {}        # base_id -> (tokens, boxes)

    sharder = StreamingParquetSharder(
        output_dir=DOCVQA_SHARD_DIR,
        split_name=split_name,
        total_rows=len(items),
        shard_size=DOCVQA_SHARD_SIZE,
        compression="zstd"
    )

    batch_buffer = []
    BUFFER_FLUSH_SIZE = 50  # Flush every 50 images to keep working memory < 100MB

    print(f"\n{'='*20} Sharding DocVQA [{split_name}]: {len(items)} rows {'='*20}")
    for item in tqdm(items, desc=f"DocVQA [{split_name}] (Lossless PNG)"):
        q_id = str(item['questionId'])
        img_rel = item['image'].lstrip("./")
        img_basename = os.path.basename(img_rel)
        base_id = os.path.splitext(img_basename)[0]

        # --- A. Fast On-Demand Image Loading (Zero RAM Accumulation) ---
        img_disk_path = DOC_IMG_DIR / img_rel
        if not img_disk_path.exists():
            img_disk_path = DOC_IMG_DIR / img_basename
            if not img_disk_path.exists():
                continue

        raw_png_bytes = img_disk_path.read_bytes()

        if img_basename not in image_dim_cache:
            with Image.open(io.BytesIO(raw_png_bytes)) as img:
                exif = img.getexif()
                orientation = exif.get(0x0112, 1) if exif else 1
                orig_w, orig_h = img.size
                if orientation not in (1, 0, None):
                    transposed = ImageOps.exif_transpose(img)
                    orig_w, orig_h = transposed.size
            image_dim_cache[img_basename] = (orig_w, orig_h, orientation)

        orig_w, orig_h, orientation = image_dim_cache[img_basename]

        if orientation not in (1, 0, None):
            with Image.open(io.BytesIO(raw_png_bytes)) as img:
                transposed = ImageOps.exif_transpose(img)
                buf = io.BytesIO()
                transposed.save(buf, format="PNG", optimize=True)
                final_png_bytes = buf.getvalue()
        else:
            final_png_bytes = raw_png_bytes

        # --- B. Fast Cached OCR Retrieval ---
        if base_id not in ocr_cache:
            ocr_tokens, ocr_boxes = [], []
            ocr_candidate = DOC_OCR_DIR / f"ocr_results/{base_id}.json"
            if not ocr_candidate.exists():
                ocr_candidate = DOC_OCR_DIR / f"{base_id}.json"

            if ocr_candidate.exists():
                with open(ocr_candidate, 'r', encoding='utf-8') as f_ocr:
                    ocr_json = json.load(f_ocr)
                    for page in ocr_json.get('recognitionResults', []):
                        for line in page.get('lines', []):
                            for w in line.get('words', []):
                                t = w.get('text', '').strip()
                                poly = w.get('boundingBox', [])
                                if t and poly:
                                    ocr_tokens.append(t)
                                    ocr_boxes.append(normalize_ms_ocr_polygon(poly, orig_w, orig_h))
            ocr_cache[base_id] = (ocr_tokens, ocr_boxes)

        ocr_tokens, ocr_boxes = ocr_cache[base_id]
        answers = item.get('answers', None)

        batch_buffer.append({
            "sample_id": f"docvqa_{q_id}",
            "domain_type": "document_text",
            "split": split_name,
            "question_id": q_id,
            "question": item['question'].strip(),
            "question_tokens": item['question'].lower().split(),
            "image_id": img_basename,
            "image": {'bytes': final_png_bytes, 'path': img_basename},
            "image_width": orig_w,
            "image_height": orig_h,
            "answers": answers if answers else [],
            "canonical_answer": extract_canonical_answer(answers),
            "ocr_tokens": ocr_tokens,
            "ocr_boxes": ocr_boxes,
            "metadata": json.dumps({
                "question_types": item.get('question_types', []),
                "docId": item.get('docId', None),
                "ucsf_document_id": item.get('ucsf_document_id', ''),
                "ucsf_document_page_no": item.get('ucsf_document_page_no', '')
            }, ensure_ascii=False)
        })

        if len(batch_buffer) >= BUFFER_FLUSH_SIZE:
            sharder.write_rows(batch_buffer)
            batch_buffer.clear()

    if batch_buffer:
        sharder.write_rows(batch_buffer)
        batch_buffer.clear()

    shards = sharder.close()
    gc.collect()
    pa.default_memory_pool().release_unused()
    print(f"✓ Completed DocVQA [{split_name}]: {len(shards)} shards created.")

process_docvqa_streaming("train", "annotations/train_v1.0_withQT.json")
process_docvqa_streaming("validation", "annotations/val_v1.0_withQT.json")
process_docvqa_streaming("test", "annotations/test_v1.0.json")


Extracting images sequentially to disk (one-time linear pass)...
  🗑️ Deleted spdocvqa_images.tar.gz to reclaim 8.6 GB disk space.
Extracting OCR sequentially to disk (one-time linear pass)...

==================== Sharding DocVQA [train]: 39463 rows ====================


DocVQA [train] (Lossless PNG):   0%|          | 0/39463 [00:00<?, ?it/s]

  ✓ Shard 1/50: train-00000-of-00050.parquet (30.7 MB)
  ✓ Shard 2/50: train-00001-of-00050.parquet (211.3 MB)
  ✓ Shard 3/50: train-00002-of-00050.parquet (188.7 MB)
  ✓ Shard 4/50: train-00003-of-00050.parquet (172.1 MB)
  ✓ Shard 5/50: train-00004-of-00050.parquet (60.3 MB)
  ✓ Shard 6/50: train-00005-of-00050.parquet (16.8 MB)
  ✓ Shard 7/50: train-00006-of-00050.parquet (17.8 MB)
  ✓ Shard 8/50: train-00007-of-00050.parquet (18.6 MB)
  ✓ Shard 9/50: train-00008-of-00050.parquet (23.3 MB)
  ✓ Shard 10/50: train-00009-of-00050.parquet (15.3 MB)
  ✓ Shard 11/50: train-00010-of-00050.parquet (20.6 MB)
  ✓ Shard 12/50: train-00011-of-00050.parquet (33.0 MB)
  ✓ Shard 13/50: train-00012-of-00050.parquet (19.9 MB)
  ✓ Shard 14/50: train-00013-of-00050.parquet (20.5 MB)
  ✓ Shard 15/50: train-00014-of-00050.parquet (29.5 MB)
  ✓ Shard 16/50: train-00015-of-00050.parquet (231.2 MB)
  ✓ Shard 17/50: train-00016-of-00050.parquet (159.3 MB)
  ✓ Shard 18/50: train-00017-of-00050.parquet (260.6

DocVQA [validation] (Lossless PNG):   0%|          | 0/5349 [00:00<?, ?it/s]

  ✓ Shard 1/7: validation-00000-of-00007.parquet (105.1 MB)
  ✓ Shard 2/7: validation-00001-of-00007.parquet (134.7 MB)
  ✓ Shard 3/7: validation-00002-of-00007.parquet (153.1 MB)
  ✓ Shard 4/7: validation-00003-of-00007.parquet (190.9 MB)
  ✓ Shard 5/7: validation-00004-of-00007.parquet (165.1 MB)
  ✓ Shard 6/7: validation-00005-of-00007.parquet (192.2 MB)
  ✓ Shard 7/7: validation-00006-of-00007.parquet (134.9 MB)
✓ Completed DocVQA [validation]: 7 shards created.

==================== Sharding DocVQA [test]: 5188 rows ====================


DocVQA [test] (Lossless PNG):   0%|          | 0/5188 [00:00<?, ?it/s]

  ✓ Shard 1/7: test-00000-of-00007.parquet (128.7 MB)
  ✓ Shard 2/7: test-00001-of-00007.parquet (142.8 MB)
  ✓ Shard 3/7: test-00002-of-00007.parquet (166.6 MB)
  ✓ Shard 4/7: test-00003-of-00007.parquet (180.9 MB)
  ✓ Shard 5/7: test-00004-of-00007.parquet (183.2 MB)
  ✓ Shard 6/7: test-00005-of-00007.parquet (208.7 MB)
  ✓ Shard 7/7: test-00006-of-00007.parquet (121.8 MB)
✓ Completed DocVQA [test]: 7 shards created.


In [8]:
# 7. Verification: Total Row Count Conservation Across All Shards
def verify_sharded_split(shard_dir: Path, split_name: str, expected_count: int):
    shards = sorted(shard_dir.glob(f"{split_name}-*.parquet"))
    assert len(shards) > 0, f"No shards found for {split_name} in {shard_dir}"

    total_rows = 0
    total_bytes = 0
    for sp in shards:
        pf = pq.ParquetFile(sp)
        total_rows += pf.metadata.num_rows
        total_bytes += sp.stat().st_size

    assert total_rows == expected_count, f"Mismatch for {split_name}: expected {expected_count}, got {total_rows}"
    print(f"  ✓ {split_name.capitalize():10s}: 100% Match ({total_rows}/{expected_count} rows across {len(shards)} shards, total {total_bytes / (1024**3):.2f} GB)")

print("=== VERIFYING TEXTVQA SHARDS ===")
verify_sharded_split(TEXTVQA_SHARD_DIR, "train", 34602)
verify_sharded_split(TEXTVQA_SHARD_DIR, "validation", 5000)
verify_sharded_split(TEXTVQA_SHARD_DIR, "test", 5734)

print("\n=== VERIFYING DOCVQA SHARDS ===")
verify_sharded_split(DOCVQA_SHARD_DIR, "train", 39463)
verify_sharded_split(DOCVQA_SHARD_DIR, "validation", 5349)
verify_sharded_split(DOCVQA_SHARD_DIR, "test", 5188)


=== VERIFYING TEXTVQA SHARDS ===
  ✓ Train     : 100% Match (34602/34602 rows across 14 shards, total 5.78 GB)
  ✓ Validation: 100% Match (5000/5000 rows across 2 shards, total 0.86 GB)
  ✓ Test      : 100% Match (5734/5734 rows across 3 shards, total 0.90 GB)

=== VERIFYING DOCVQA SHARDS ===
  ✓ Train     : 100% Match (39463/39463 rows across 50 shards, total 7.30 GB)
  ✓ Validation: 100% Match (5349/5349 rows across 7 shards, total 1.05 GB)
  ✓ Test      : 100% Match (5188/5188 rows across 7 shards, total 1.11 GB)


In [9]:
# 8. Upload Sharded Repositories to Hugging Face Hub (Lossless Branch) & Clean Up
def upload_sharded_canonical_repo(repo_name, shard_dir, task_category, tags, branch="lossless", clean_local=False):
    full_repo = f"{username}/{repo_name}"
    api.create_repo(repo_id=full_repo, repo_type="dataset", exist_ok=True)

    # 1. Ensure target branch exists
    api.create_branch(repo_id=full_repo, repo_type="dataset", branch=branch, exist_ok=True)
    print(f"\n{'='*25} Syncing {full_repo} (Branch: {branch}) {'='*25}")

    # 2. Write README.md with wildcard split matching
    readme_content = f"""---
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train-*
  - split: validation
    path: data/validation-*
  - split: test
    path: data/test-*
task_categories:
- {task_category}
tags:
{tags}
---
# {repo_name} (Lossless Sharded Canonical)
Normalized Canonical Dataset for CrossVQA Benchmark ({branch} branch).
- **Format preservation:** 100% original image formats (Lossless PNG for DocVQA, Native JPEG for TextVQA).
- **Streaming Shards:** Optimized with page index (`write_page_index=True`) for instant browsing on Hugging Face Dataset Viewer without timeouts.
- **Spatial & EXIF Alignment:** All OCR boxes scaled to [ymin, xmin, ymax, xmax] in [0, 1000].
"""
    readme_path = WORK_DIR / f"README_{repo_name}_{branch}.md"
    readme_path.write_text(readme_content, encoding='utf-8')
    api.upload_file(
        path_or_fileobj=str(readme_path),
        path_in_repo="README.md",
        repo_id=full_repo,
        repo_type="dataset",
        revision=branch
    )
    print("  ✓ Updated README.md configuration metadata.")

    # 3. Clean up obsolete shards inherited from 'main' (e.g. DocVQA's 25 *-of-00027.parquet JPEG shards)
    # This prevents glob collisions in Dataset Viewer where both old and new shards would be read.
    all_new_shards = sorted(shard_dir.glob("*.parquet"))
    new_shard_names = {f"data/{s.name}" for s in all_new_shards}

    try:
        remote_repo_files = api.list_repo_files(repo_id=full_repo, repo_type="dataset", revision=branch)
        remote_data_files = [f for f in remote_repo_files if f.startswith("data/") and f.endswith(".parquet")]
        obsolete_files = [f for f in remote_data_files if f not in new_shard_names]

        if obsolete_files:
            print(f"  🗑️ Detected {len(obsolete_files)} obsolete inherited shards on '{branch}' branch.")
            print(f"     Pruning them via atomic commit to prevent duplicate row errors...")
            delete_ops = [CommitOperationDelete(path_in_repo=f) for f in obsolete_files]
            api.create_commit(
                repo_id=full_repo,
                repo_type="dataset",
                revision=branch,
                operations=delete_ops,
                commit_message="Prune obsolete shards inherited from main branch"
            )
            print(f"  ✓ Successfully pruned {len(obsolete_files)} obsolete shards.")
        else:
            print("  ✓ No obsolete shards detected on remote branch.")
    except Exception as e:
        print(f"  Warning during obsolete shard pruning: {e}")

    # 4. Upload all newly generated lossless shards (with delete_patterns to ensure perfect sync)
    print(f"  Uploading {len(all_new_shards)} shards from {shard_dir.name} to {full_repo} (branch: {branch})...")
    api.upload_folder(
        folder_path=str(shard_dir),
        path_in_repo="data",
        repo_id=full_repo,
        repo_type="dataset",
        revision=branch,
        delete_patterns="*.parquet"
    )
    print(f"✓ Successfully synced {len(all_new_shards)} shards to '{branch}' branch:")
    print(f"  https://huggingface.co/datasets/{full_repo}/tree/{branch}")

    # 5. Optional local disk cleanup (reclaims ~50 GB in Colab after successful upload)
    if clean_local:
        shutil.rmtree(shard_dir, ignore_errors=True)
        print(f"  🧹 Cleaned local shards to free disk space: {shard_dir}")

# Upload TextVQA Sharded Canonical (Lossless Branch)
upload_sharded_canonical_repo(
    repo_name="textvqa-canonical",
    shard_dir=TEXTVQA_SHARD_DIR,
    task_category="visual-question-answering",
    tags="- textvqa\n- scene-text\n- canonical\n- crossvqa\n- lossless\n- sharded",
    branch=TARGET_BRANCH,
    clean_local=False  # Set to True on Colab if low on disk
)

# Upload DocVQA Sharded Canonical (Lossless Branch)
upload_sharded_canonical_repo(
    repo_name="docvqa-canonical",
    shard_dir=DOCVQA_SHARD_DIR,
    task_category="visual-question-answering",
    tags="- docvqa\n- document-ai\n- canonical\n- crossvqa\n- lossless\n- sharded",
    branch=TARGET_BRANCH,
    clean_local=False  # Set to True on Colab if low on disk
)



========================= Syncing gamusa/textvqa-canonical (Branch: lossless) =========================
  ✓ Updated README.md configuration metadata.
  ✓ No obsolete shards detected on remote branch.
  Uploading 19 shards from textvqa to gamusa/textvqa-canonical (branch: lossless)...
✓ Successfully synced 19 shards to 'lossless' branch:
  https://huggingface.co/datasets/gamusa/textvqa-canonical/tree/lossless

========================= Syncing gamusa/docvqa-canonical (Branch: lossless) =========================
  ✓ Updated README.md configuration metadata.
  🗑️ Detected 33 obsolete inherited shards on 'lossless' branch.
     Pruning them via atomic commit to prevent duplicate row errors...
  ✓ Successfully pruned 33 obsolete shards.
  Uploading 64 shards from docvqa to gamusa/docvqa-canonical (branch: lossless)...
✓ Successfully synced 64 shards to 'lossless' branch:
  https://huggingface.co/datasets/gamusa/docvqa-canonical/tree/lossless


In [10]:
# 9. Visual Inspection & Format Verification
import io
import pyarrow.parquet as pq
from pathlib import Path
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

def display_first_5_images_from_shards(shard_dir: Path, split_name: str):
    first_shard = sorted(shard_dir.glob(f"{split_name}-*.parquet"))[0]
    print(f"\n{'='*20} First 5 Images from {shard_dir.name}/{first_shard.name} {'='*20}")

    pfile = pq.ParquetFile(first_shard)
    batch_iter = pfile.iter_batches(batch_size=5, columns=['image', 'question_id'])
    first_batch = next(batch_iter)
    df = first_batch.to_pandas()

    images = []
    titles = []

    for idx, row in df.iterrows():
        img_data = row['image']
        if isinstance(img_data, dict) and 'bytes' in img_data:
            img_bytes = img_data['bytes']
            img = Image.open(io.BytesIO(img_bytes))
            fmt = img.format
            mode = img.mode
            sz = img.size
            print(f"  Row {idx} ({row['question_id']}): Format={fmt}, Size={sz}, Mode={mode}, Payload={len(img_bytes)/1024:.1f} KB")

            thumb = img.copy()
            thumb.thumbnail((250, 250))
            images.append(thumb)
            titles.append(f"{row['question_id']} [{fmt}]")

    out_images = []
    for img, title in zip(images, titles):
        buf = io.BytesIO()
        img.save(buf, format="PNG" if img.format == "PNG" else "JPEG")
        img_widget = widgets.Image(value=buf.getvalue(), width=180, height=180)
        box = widgets.VBox([widgets.Label(value=title), img_widget])
        out_images.append(box)

    display(widgets.HBox(out_images))

# Display for both datasets from validation shards
display_first_5_images_from_shards(TEXTVQA_SHARD_DIR, "validation")
display_first_5_images_from_shards(DOCVQA_SHARD_DIR, "validation")



==================== First 5 Images from textvqa/validation-00000-of-00002.parquet ====================
  Row 0 (34602): Format=JPEG, Size=(1024, 664), Mode=RGB, Payload=42.4 KB
  Row 1 (34603): Format=JPEG, Size=(1024, 683), Mode=RGB, Payload=299.6 KB
  Row 2 (34604): Format=JPEG, Size=(1024, 1024), Mode=RGB, Payload=169.9 KB
  Row 3 (34605): Format=JPEG, Size=(1024, 576), Mode=RGB, Payload=251.0 KB
  Row 4 (34606): Format=JPEG, Size=(1024, 576), Mode=RGB, Payload=251.0 KB



==================== First 5 Images from docvqa/validation-00000-of-00007.parquet ====================
  Row 0 (49153): Format=PNG, Size=(2257, 1764), Mode=L, Payload=1277.8 KB
  Row 1 (24580): Format=PNG, Size=(808, 1077), Mode=L, Payload=90.2 KB
  Row 2 (57349): Format=PNG, Size=(1701, 2386), Mode=L, Payload=757.6 KB
  Row 3 (24581): Format=PNG, Size=(808, 1077), Mode=L, Payload=90.2 KB
  Row 4 (24582): Format=PNG, Size=(808, 1077), Mode=L, Payload=90.2 KB


### Explore one parquet shard

In [11]:
import pyarrow.parquet as pq
import pandas as pd

# Path to the selected Parquet file
parquet_file_path = "/content/crossvqa_lossless_workspace/shards/docvqa/train-00000-of-00050.parquet"

# 1. Load Parquet Metadata and Schema
print("=== PARQUET SCHEMA ===")
pf = pq.ParquetFile(parquet_file_path)
print(pf.schema)

# 2. Preview First 10 Rows (excluding raw image bytes for cleaner output)
print("\n=== PREVIEW FIRST 10 ROWS (Metadata & OCR) ===")
table = pf.read_row_group(0) # Read first row group
df = table.to_pandas().head(10)

# Exclude raw image bytes from printing so it doesn't clutter the stdout
if 'image' in df.columns:
    df_preview = df.copy()
    df_preview['image'] = df_preview['image'].apply(lambda x: f"<PNG Bytes: {len(x['bytes'])} bytes>" if isinstance(x, dict) and 'bytes' in x else x)
    display(df_preview)
else:
    display(df)

# 3. Print basic description
print("\n=== SHARD SUMMARY ===")
print(f"File path: {parquet_file_path}")
print(f"Total rows in shard: {pf.metadata.num_rows}")
print(f"Number of columns: {pf.metadata.num_columns}")
print(f"Serialized size: {sum(pf.metadata.row_group(i).total_byte_size for i in range(pf.metadata.num_row_groups)) / (1024**2):.2f} MB")

=== PARQUET SCHEMA ===
required group field_id=-1 schema {
  optional binary field_id=-1 sample_id (String);
  optional binary field_id=-1 domain_type (String);
  optional binary field_id=-1 split (String);
  optional binary field_id=-1 question_id (String);
  optional binary field_id=-1 question (String);
  optional group field_id=-1 question_tokens (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional binary field_id=-1 image_id (String);
  optional group field_id=-1 image {
    optional binary field_id=-1 bytes;
    optional binary field_id=-1 path (String);
  }
  optional int64 field_id=-1 image_width;
  optional int64 field_id=-1 image_height;
  optional group field_id=-1 answers (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional binary field_id=-1 canonical_answer (String);
  optional group field_id=-1 ocr_tokens (List) {
    repeated group field_

,sample_id,domain_type,split,question_id,question,question_tokens,image_id,image,image_width,image_height,answers,canonical_answer,ocr_tokens,ocr_boxes,metadata
0,docvqa_337,document_text,train,337,what is the date mentioned in this letter?,"[what, is, the, date, mentioned, in, this, let...",xnbl0037_1.png,<PNG Bytes: 57498 bytes>,1695,2025,[1/8/93],1/8/93,"[Confidential, .., .., RJRT, PR, APPROVAL, DAT...","[[15, 345, 65, 586], [16, 637, 27, 648], [15, ...","{""question_types"": [""handwritten"", ""form""], ""d..."
1,docvqa_338,document_text,train,338,what is the contact person name mentioned in l...,"[what, is, the, contact, person, name, mention...",xnbl0037_1.png,<PNG Bytes: 57498 bytes>,1695,2025,"[P. Carter, p. carter]",P. Carter,"[Confidential, .., .., RJRT, PR, APPROVAL, DAT...","[[15, 345, 65, 586], [16, 637, 27, 648], [15, ...","{""question_types"": [""handwritten"", ""form""], ""d..."
2,docvqa_339,document_text,train,339,Which corporation's letterhead is this?,"[which, corporation's, letterhead, is, this?]",mxcj0037_1.png,<PNG Bytes: 220511 bytes>,1606,2104,[Brown & Williamson Tobacco Corporation],Brown & Williamson Tobacco Corporation,"[B&W, BROWN, &, WILLIAMSON, TOBACCO, CORPORATI...","[[92, 456, 104, 493], [119, 297, 131, 353], [1...","{""question_types"": [""layout""], ""docId"": 280, ""..."
3,docvqa_340,document_text,train,340,Who is in cc in this letter?,"[who, is, in, cc, in, this, letter?]",mxcj0037_1.png,<PNG Bytes: 220511 bytes>,1606,2104,[T.F. Riehl],T.F. Riehl,"[B&W, BROWN, &, WILLIAMSON, TOBACCO, CORPORATI...","[[92, 456, 104, 493], [119, 297, 131, 353], [1...","{""question_types"": [""form"", ""layout""], ""docId""..."
4,docvqa_341,document_text,train,341,what is the subject of this letter?,"[what, is, the, subject, of, this, letter?]",mxcj0037_1.png,<PNG Bytes: 220511 bytes>,1606,2104,[Review of existing Brainstorming Ideas/483],Review of existing Brainstorming Ideas/483,"[B&W, BROWN, &, WILLIAMSON, TOBACCO, CORPORATI...","[[92, 456, 104, 493], [119, 297, 131, 353], [1...","{""question_types"": [""form"", ""layout""], ""docId""..."
5,docvqa_343,document_text,train,343,"What is the number at the bottom of the page, ...","[what, is, the, number, at, the, bottom, of, t...",mxcj0037_1.png,<PNG Bytes: 220511 bytes>,1606,2104,[499150498],499150498,"[B&W, BROWN, &, WILLIAMSON, TOBACCO, CORPORATI...","[[92, 456, 104, 493], [119, 297, 131, 353], [1...","{""question_types"": [""layout""], ""docId"": 280, ""..."
6,docvqa_388,document_text,train,388,How many points are there in modifications to ...,"[how, many, points, are, there, in, modificati...",sxxj0037_2.png,<PNG Bytes: 77687 bytes>,1692,2245,"[5., 5]",5.,"[2., The, rotometer, in, the, original, design...","[[135, 202, 147, 215], [135, 245, 150, 272], [...","{""question_types"": [""table/list""], ""docId"": 29..."
7,docvqa_399,document_text,train,399,what is the date in the letter,"[what, is, the, date, in, the, letter]",nhxj0037_1.png,<PNG Bytes: 133639 bytes>,1706,2198,"[June 11, 1990, June 11,1990]","June 11, 1990","[PHILIP, MORRIS, U., S., A., INTER-OFFICE, COR...","[[81, 348, 96, 410], [81, 413, 96, 493], [81, ...","{""question_types"": [""form"", ""layout""], ""docId""..."
8,docvqa_400,document_text,train,400,To whom this is addressed,"[to, whom, this, is, addressed]",nhxj0037_1.png,<PNG Bytes: 133639 bytes>,1706,2198,[Dr.K.S.Houghton],Dr.K.S.Houghton,"[PHILIP, MORRIS, U., S., A., INTER-OFFICE, COR...","[[81, 348, 96, 410], [81, 413, 96, 493], [81, ...","{""question_types"": [""form"", ""layout""], ""docId""..."
9,docvqa_401,document_text,train,401,Who sent the letter?,"[who, sent, the, letter?]",nhxj0037_1.png,<PNG Bytes: 133639 bytes>,1706,2198,[Ted Sanders],Ted Sanders,"[PHILIP, MORRIS, U., S., A., INTER-OFFICE, COR...","[[81, 348, 96, 410], [81, 413, 96, 493], [81, ...","{""question_types"": [""form"", ""layout""], ""docId""..."



=== SHARD SUMMARY ===
File path: /content/crossvqa_lossless_workspace/shards/docvqa/train-00000-of-00050.parquet
Total rows in shard: 800
Number of columns: 16
Serialized size: 32.64 MB
